# Preparation

Run `pip install -r requirements.txt`, then run this notebook.

# Initial set up

I'm importing packages and defining some globals for the notebook here. I set a random seed for reproducibility.

> **AI usage disclaimer:** I've come to embrace AI-assisted development, the work presented in this notebook has been partially generated using LLMs, however, I've supervised the entire process and ultimately decided which parts to change, update, and customize. This disclaimer is an indicator of my commitment to ethical, responsible, and transparent use of AI. All the writing (unless stated) has been done by me, as a demonstration of my understanding of the problem and the chosen approach; also, this project has been built gradually using multiple commits so it evidences my work.

> **Execution disclaimer:** I own an RTX 5090 GPU, which I was unable to use due to some issues with my OS, so I was limited to use a MacBook pro with an M1 Max and 32 GB of memory, so CUDA behavior has not been tested. I use a smaller model for MPS/CPU and a larger one if CUDA is available.

In [ ]:
import json
import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

from datasets import Dataset

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTConfig, SFTTrainer

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

IS_CUDA = torch.cuda.is_available()
IS_MPS = torch.backends.mps.is_available()

if IS_CUDA:
    DEVICE = "cuda"
    MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
elif IS_MPS:
    DEVICE = "mps"
    MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
else:
    DEVICE = "cpu"
    MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

OUTPUT_DIR = "qwen3-vl-health-table-extraction-lora"
print(f"Device: {DEVICE} | IS_CUDA={IS_CUDA} | Model: {MODEL_ID}")

# Overall description

Since we are dealing with complex (and variable) form layouts, we have to represent this data in a structured way. The most natural representation (with many benchmarks available as well) is with standard HTML tables, which allow us to make use of rowspan and colspan to merge cells and create fairly complex tables in a digital form.

**The HTML table** is wrapped in a JSON envelope that carries key/value fields (patient identifiers *redacted for training*, form date, form type):

**AI generated sample**

```json
{
  "form_type": "basic_metabolic_panel",
  "fields": {"specimen_date": "2026-03-14", "ordering_provider": "REDACTED"},
  "table_html": "<table><tr><th>Test</th><th colspan=\"2\">Reference Range</th><th>Result</th></tr>...</table>"
}
```

The model's job is: given the form image, emit exactly this JSON object as a string, while still getting HTML's structural expressiveness for the table body.

# Synthetic training data [AI generated helper]

In [ ]:
def _render_synthetic_lab_form(path: Path, seed: int) -> dict:
    """Render a crude 'lab panel' table into a high-resolution PNG and return
    the matching ground-truth record. This stands in for a real, scanned
    health-form image + a human- or vendor-labeled structured target.

    In production this function is replaced entirely: images come from your
    document ingestion pipeline (scanner/fax/EHR export), and targets come
    from human annotation (e.g., via Label Studio with an HTML-table export
    plugin) or from a trusted upstream structured source used to
    bootstrap weak labels.
    """
    rng = random.Random(seed)
    tests = [
        ("Sodium", "136", "145", "mmol/L"),
        ("Potassium", "3.5", "5.1", "mmol/L"),
        ("Chloride", "98", "107", "mmol/L"),
        ("Glucose", "70", "99", "mg/dL"),
        ("BUN", "7", "20", "mg/dL"),
        ("Creatinine", "0.6", "1.3", "mg/dL"),
    ]

    # High-resolution canvas: scanned health forms are frequently
    # 2000-3000px on the long edge. We deliberately render at high res so
    # the notebook exercises the same resolution regime as real scans.
    width, height = 1700, 2200
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    draw.text((60, 60), "COMPREHENSIVE METABOLIC PANEL", fill="black", font=font)
    draw.text((60, 110), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)

    top = 220
    row_h = 70
    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    for x, h in zip(col_x, headers):
        draw.text((x, top), h, fill="black", font=font)
    draw.line([(60, top + 50), (1640, top + 50)], fill="black", width=3)

    rows_html = []
    for i, (name, lo, hi, unit) in enumerate(tests):
        y = top + 70 + i * row_h
        result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
        row_vals = [name, result, lo, hi, unit]
        for x, v in zip(col_x, row_vals):
            draw.text((x, y), v, fill="black", font=font_small)
        rows_html.append(
            "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
        )

    img.save(path)

    table_html = (
        "<table>"
        "<tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
        + "".join(rows_html)
        + "</table>"
    )
    target = {
        "form_type": "basic_metabolic_panel",
        "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
        "table_html": table_html,
    }
    return {"image": str(path), "target": target}


def build_synthetic_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"form_{i:04d}.png"
        record = _render_synthetic_lab_form(img_path, seed=i)
        manifest.append(record)
    return manifest


synthetic_manifest = build_synthetic_manifest(n_examples=24, out_dir="./synthetic_health_forms")
print(f"Built {len(synthetic_manifest)} synthetic (image, target) pairs.")
print(json.dumps(synthetic_manifest[0], indent=2)[:600], "...")


# Qwen3-VL chat template

Qwen3-VL is instruction-tuned and expects input as a **list of role-tagged messages**, where each message's `content` is itself a list of typed content blocks (`{"type": "text", "text": ...}` or `{"type": "image", "image": ...}`).

For training, **three** messages are constructed per example: System message (general instructions), user turn (image and brief description), system turn (processing output used as ground truth).

The training consists on masking the ground truth and comparing the output to compute loss.


In [ ]:
SYSTEM_PROMPT = (
    "You are a meticulous medical document structuring assistant. You are shown an image "
    "of a health form that contains one or more tables. Your job is to transcribe the form's "
    "table(s) and a small set of header fields into a single JSON object with exactly this shape:\n\n"
    '{"form_type": <string>, "fields": {"specimen_date": <string or null>, '
    '"ordering_provider": <string or null>}, "table_html": <string>}\n\n'
    "Rules:\n"
    "1. `table_html` must be a valid, minimal HTML <table> using <tr>/<th>/<td> and, when the form "
    "uses merged cells, `rowspan`/`colspan` attributes that faithfully reproduce the visual layout.\n"
    "2. Transcribe every visible row and column; do not summarize, reorder, or drop rows.\n"
    "3. If a field is not present on the form, use null rather than guessing.\n"
    "4. Output ONLY the JSON object. No markdown fences, no commentary.\n"
    "5. Never fabricate patient-identifying values; if a field is redacted or illegible, output null."
)

USER_INSTRUCTION = "Extract this form's structured data as JSON, per the schema and rules above."

# AI generated
def format_example(record: dict) -> list[dict]:
    """Convert one manifest record into the 3-turn chat-message structure
    Qwen3-VL's processor expects.

    `record["image"]` may be a filesystem path, a URL, or a PIL.Image — the
    `qwen_vl_utils.process_vision_info` helper accepts all three inside the
    `{"type": "image", "image": ...}` block, which is convenient: we don't
    have to eagerly load every image into memory just to build the message
    list.
    """
    target_str = json.dumps(record["target"], ensure_ascii=False)
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": record["image"]},
                {"type": "text", "text": USER_INSTRUCTION},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": target_str}],
        },
    ]


formatted_examples = [format_example(r) for r in synthetic_manifest]
print(formatted_examples[0][1])   # the user turn
print()
print(formatted_examples[0][2])   # the assistant turn


# Visual tokens budget

To account for high resolution images and prevent hallucinations caused by a model incapable of reading the image content we set a processor that automatically scales the image resolution to the nearest ViT grid:

MIN_PIXELS <= H*W <= MAX_PIXELS


In [ ]:
# Hyperparameter tuning could be beneficial here if in scope to determine the best max pixels value within budget, 2048 has been used as an arbitrarily high value.
MIN_PIXELS = 256 * 32 * 32
MAX_PIXELS = 2048 * 32 * 32

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
print(processor)


# Batch input + mask tokens for inference [AI generated]

Tokenize the *prompt-only* rendering (system + user turn, with the generation prompt opened) separately from the *full* rendering (system + user + assistant turn), and mask everything up to the length of the prompt-only tokenization.

In [ ]:
IMAGE_TOKEN_STRINGS = ["<|vision_start|>", "<|vision_end|>", "<|image_pad|>"]

def _image_token_ids(proc) -> list[int | None]:
    ids = [proc.tokenizer.convert_tokens_to_ids(t) for t in IMAGE_TOKEN_STRINGS]
    resolved = [i for i in ids if i is not None and i >= 0]
    if len(resolved) != len(IMAGE_TOKEN_STRINGS):
        print(
            "WARNING: not all image special tokens resolved to an id "
            f"({resolved} from {IMAGE_TOKEN_STRINGS}). Inspect "
            "proc.tokenizer.special_tokens_map / additional_special_tokens "
            "before trusting the collator's loss masking."
        )
    return resolved


IMAGE_TOKEN_IDS = _image_token_ids(processor)
print("Image-related special token ids:", IMAGE_TOKEN_IDS)


def collate_fn(examples: list[list[dict]]) -> dict[str, torch.Tensor]:
    """Custom multimodal collator for trl.SFTTrainer.

    `examples` is a list of chat-message lists (system/user/assistant), i.e.
    exactly what `format_example()` produces. Each call receives one
    mini-batch worth of examples.
    """
    full_texts, prompt_texts, image_inputs = [], [], []

    for messages in examples:
        # Full rendering: system + user + assistant -> this is the sequence used for the actual training.
        full_texts.append(
            processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        )
        # Prompt-only rendering: system + user, with the generation prompt
        # opened -> used purely to measure how many leading tokens to mask.
        prompt_texts.append(
            processor.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
        )
        imgs, _ = process_vision_info(messages)
        image_inputs.append(imgs[0] if imgs else None)

    batch = processor(
        text=full_texts,
        images=image_inputs,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()

    # 1) Mask padding.
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # 2) Mask image-placeholder / vision-boundary tokens -- these carry no
    #    linguistic content to predict.
    for image_token_id in IMAGE_TOKEN_IDS:
        labels[labels == image_token_id] = -100

    # 3) Mask the prompt portion (system + user turn) of every sequence, so
    #    loss is computed only on the assistant's structured-output tokens.
    #    We recompute the prompt length per-example with the SAME tokenizer
    #    call path (tokenize=True) so left-padding side / truncation cannot
    #    silently desync the boundary from the batch's actual input_ids.
    for i, prompt_text in enumerate(prompt_texts):
        prompt_len = len(processor.tokenizer(prompt_text, add_special_tokens=False).input_ids)
        labels[i, :prompt_len] = -100

    batch["labels"] = labels
    return batch


# Multi-table synthetic data [AI generated helper]

In [ ]:
def _render_synthetic_multi_table_form(path: Path, seed: int) -> dict:
    """Render a page containing TWO lab panels stacked vertically, and
    return one record per panel with that panel's exact pixel bounding box
    (in the *original, full-resolution* image) plus its structured target.

    This stands in for a real multi-section health form (e.g., a combined
    metabolic + hematology requisition) where more than one table shares a
    page. As in Section 1.2, in production these boxes come from your
    annotation tool, not from a rendering script -- what matters is that the
    training data preserves that mapping between a table's location and its
    transcription target.
    """
    rng = random.Random(seed)
    panels = [
        ("BASIC METABOLIC PANEL", "basic_metabolic_panel", [
            ("Sodium", "136", "145", "mmol/L"),
            ("Potassium", "3.5", "5.1", "mmol/L"),
            ("Chloride", "98", "107", "mmol/L"),
            ("Glucose", "70", "99", "mg/dL"),
        ]),
        ("COMPLETE BLOOD COUNT", "complete_blood_count", [
            ("WBC", "4.5", "11.0", "10^3/uL"),
            ("Hemoglobin", "13.5", "17.5", "g/dL"),
            ("Platelets", "150", "400", "10^3/uL"),
        ]),
    ]

    width, height = 1700, 2600
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    row_h = 70
    regions = []
    y_cursor = 60

    for title, form_type, tests in panels:
        panel_top = y_cursor
        draw.text((60, y_cursor), title, fill="black", font=font)
        y_cursor += 50
        draw.text((60, y_cursor), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)
        y_cursor += 60

        header_top = y_cursor
        for x, h in zip(col_x, headers):
            draw.text((x, y_cursor), h, fill="black", font=font)
        draw.line([(60, y_cursor + 50), (1640, y_cursor + 50)], fill="black", width=3)
        y_cursor += 70

        rows_html = []
        for name, lo, hi, unit in tests:
            result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
            row_vals = [name, result, lo, hi, unit]
            for x, v in zip(col_x, row_vals):
                draw.text((x, y_cursor), v, fill="black", font=font_small)
            rows_html.append(
                "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
            )
            y_cursor += row_h

        panel_bottom = y_cursor + 20
        # bbox convention: [x1, y1, x2, y2] in the ORIGINAL image's pixel
        # space, covering the panel's title through its last table row.
        regions.append({
            "bbox": [40, panel_top - 10, 1660, panel_bottom],
            "label": form_type,
            "target": {
                "form_type": form_type,
                "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
                "table_html": (
                    "<table><tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
                    + "".join(rows_html) + "</table>"
                ),
            },
        })
        y_cursor = panel_bottom + 80  # gap before next panel

    img.save(path)
    return {"image": str(path), "regions": regions}


def build_multi_table_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"multi_form_{i:04d}.png"
        manifest.append(_render_synthetic_multi_table_form(img_path, seed=i))
    return manifest


multi_table_manifest = build_multi_table_manifest(n_examples=12, out_dir="./synthetic_multi_table_forms")
print(f"Built {len(multi_table_manifest)} multi-table pages, "
      f"{sum(len(r['regions']) for r in multi_table_manifest)} table regions total.")
print(json.dumps(multi_table_manifest[0]['regions'][0], indent=2)[:400], "...")


# Detection (table boundaries) and transcription examples set up

The model has been trained as a general purpose tool, so the objective is to "improve" its framing (table detection) and transcription (character recognition in cropped images) capabilities through the usage of a special set of examples.

## Detection examples

The image containing the tables is resized to a smaller pixel budget since this part of the training is for inducing some "shape detection" capabilities into the model.

The idea is for the model to be able to detect the table boundaries and pick a set of coordinates (x1, y1) and (x2, y2) to locate a table. These coordinates are normalized to the range [0,1000] which is relative to the image's Width and Height. This normalization is because Qwen3-VL uses a relative 1000×1000 coordinate grid for its 2D grounding and bounding box generation, regardless of the image’s original aspect ratio or absolute pixel dimensions.

## Transcription examples

The original image (full resolution) is cropped using the detected boundaries plus a small fractional margin (or the image boundaries if the margin goes beyond these).

The model should be able to scan the text in the cropped region and output the HTML-embedded JSON format defined earlier.

# Objective

Given these two specialized tasks, we can proceed with a special training loop, where we run the tasks, compute the loss, and fine-tune.

In [ ]:
DETECTION_SYSTEM_PROMPT = (
    "You are a document layout analysis assistant. You are shown an image of a health form page "
    "that may contain one or more tables. Locate every table on the page and output ONLY a JSON "
    "list, one entry per table, of the form "
    '{"bbox_2d": [x1, y1, x2, y2], "label": <short form-type slug>}. '
    "Coordinates are on a NORMALIZED 0-1000 scale relative to this image's width and height "
    "(NOT raw pixel values): x1,x2 in [0,1000] represent the fraction of the image WIDTH, "
    "y1,y2 in [0,1000] represent the fraction of the image HEIGHT. [x1, y1] is the top-left corner "
    "and [x2, y2] the bottom-right corner, and the box must fully contain the table's title and "
    "every row. Output nothing else."
)
DETECTION_INSTRUCTION = "Detect every table on this page and return their bounding boxes as specified."

# Coarser than the MIN_PIXELS/MAX_PIXELS used before since localization doesn't require much detail
# so this has a lower budget to use less resources
DETECTION_MIN_PIXELS = 256 * 32 * 32
DETECTION_MAX_PIXELS = 640 * 32 * 32


def _to_normalized_1000(x1, y1, x2, y2, orig_w, orig_h):
    """Convert an absolute-pixel bbox in the ORIGINAL image to Qwen3-VL's
    normalized 0-1000 scale."""
    return [
        round(x1 / orig_w * 1000),
        round(y1 / orig_h * 1000),
        round(x2 / orig_w * 1000),
        round(y2 / orig_h * 1000),
    ]


def build_detection_example(record: dict) -> list[dict]:
    img = Image.open(record["image"])
    orig_w, orig_h = img.size

    target_regions = []
    for region in record["regions"]:
        x1, y1, x2, y2 = region["bbox"]
        target_regions.append({
            "bbox_2d": _to_normalized_1000(x1, y1, x2, y2, orig_w, orig_h),
            "label": region["label"],
        })

    return [
        {"role": "system", "content": [{"type": "text", "text": DETECTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": record["image"]},
            {"type": "text", "text": DETECTION_INSTRUCTION},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": json.dumps(target_regions)}]},
    ]


CROP_MARGIN_FRAC = 0.03  # to avoid clipping a table's border/title


def crop_region(image: Image.Image, bbox: list[float], margin_frac: float = CROP_MARGIN_FRAC) -> Image.Image:
    W, H = image.size
    x1, y1, x2, y2 = bbox
    mx, my = (x2 - x1) * margin_frac, (y2 - y1) * margin_frac
    x1, y1 = max(0, x1 - mx), max(0, y1 - my)
    x2, y2 = min(W, x2 + mx), min(H, y2 + my)
    return image.crop((int(x1), int(y1), int(x2), int(y2)))


def build_transcription_example(record: dict, region: dict) -> list[dict]:
    img = Image.open(record["image"])
    cropped = crop_region(img, region["bbox"])
    target_str = json.dumps(region["target"], ensure_ascii=False)
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": cropped},
            {"type": "text", "text": USER_INSTRUCTION},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": target_str}]},
    ]


detection_examples = [build_detection_example(r) for r in multi_table_manifest]
transcription_examples = [
    build_transcription_example(r, region)
    for r in multi_table_manifest
    for region in r["regions"]
]
print(f"{len(detection_examples)} detection examples, {len(transcription_examples)} cropped transcription examples.")
print(detection_examples[0][2])  # the detection target for one page (bbox_2d values should now be in [0,1000])


# Training considerations

## Maximizing the model's performance

There's one important detail to consider: since the training for transcription is performed on cropped images, inference will perform better on cropped images as well. This is a limitation that resembles human assets: we have several specialized employees instead of a single employee that does his job in a mediocre way.

## Tiling

Also, in many cases te tables will fit this detect + zoom + process approach, however, there can be other cases where tables are simply too big to fit on a single crop. Essentially, this forces us to perform some sort of tiling or sliding-window across the table, while also preserving the intended table structure.

To guarantee the preservation of the structure, this tiling process should target row bands (avoiding splits mid-row or mid-column), and allowing some overlap to piece everything together while deduplicating (think of this as solving a jigsaw puzzle).


# Model + PEFT

Tensors are data structures that store numbers with a determined precision. Normally we encounter models in half precision (float16) or in bf16.

> `Qwen3-VL-8B-Instruct` has roughly 9B total parameters (8B-class language backbone + ViT + merger/DeepStack fusion layers). In bf16, weights alone are ~18GB. A **full** fine-tune with AdamW needs, per trainable parameter: 2 bytes (bf16 weight) + 2 bytes (bf16 gradient) + 8 bytes (fp32 first+second moment in Adam) = 12 bytes, i.e. ~108GB just for optimizer state and gradients on top of the 18GB of weights — before activations. That's multi-A100 territory before you've fine-tuned anything. [AI generated]

By leveraging Quantized Low-Rank Adaptation (QLoRA), we can reduce the amount of memory needed during inference, but we can also quantize the model and tune a new set of parameters while freezing these quantized parameters. Quantization reduces precision, but this precision sacrifice is minimal during inference, training is where precision is needed (due to the gradient-based optimization).

# BitsAndBytes

We target 4-bit quantization (a large model with a strong 4-bit quantization is preferred over small models with weaker quantizations).

For Qwen-VL models, the recommended quantization type is NF4 (Normal Float 4), as the community agrees that it has a better theoretical performance and higher precision for weights initialized from a normal distribution compared to standard FP4.

In [ ]:
if IS_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, # to save more VRAM with relatively low quality cost
        bnb_4bit_compute_dtype=torch.bfloat16, # to compute using bf16
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    bnb_config = None
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

# min_pixels/max_pixels is already contained in the processor
model.config.use_cache = False  # re-enabled for inference later
print(model.config.torch_dtype, next(model.parameters()).dtype)

# Gradient checkpointing

We trade reduced memory usage for increased taining times.

In [ ]:
if IS_CUDA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

#